# Classifying BBQ product reviews with Jev

A fictional barbecue retailer: customers, their orders over time, and the reviews they chose to write, generated deterministically (see [`docs/data-model.md`](../docs/data-model.md)). Every review sentence is then classified by [TypeSafe AI's Jev](https://docs.typesafe.ai/introduction), a *System One* model that answers typed questions (`Choice`, `Score`, `Noul`) with probabilities and a confidence, instead of generating text to be parsed.

1. Generate customers, orders and reviews, and write the review text.
2. Load the reviews, each with what we knew about the customer **at the moment they wrote it**.
3. Break each review into sentences.
4. Ask Jev about every sentence, with the whole review as context.
5. Roll the answers up into things a product or support team can act on.

All the logic lives in `src/`; this notebook only runs it. `TYPESAFE_API_KEY`, and the Key Vault holding the Azure AI Foundry settings that write review text, are read from `.env`.

In [1]:
from datetime import datetime
from pathlib import Path

import polars as pl
from dotenv import load_dotenv

from jev_classifier import FRUSTRATION_LEVELS, KEY, PROBLEM_CATEGORIES, QUESTION_SET_VERSION, build_questions, classify_sentences, transcript
from retail_generator import add_customers, init_dataset, status
from retail_model import DEFAULT_DIR, read_dataset
from review_insights import cross_mentions, flagged, frustration_by_rating, needs_review, problems_by_product, review_rollup
from review_wrangler import load_reviews, product_catalogue, split_sentences
from review_writer import FoundryReviewWriter

load_dotenv(Path.cwd().parent / ".env")
OUTPUT = Path.cwd().parent / "data" / "output"
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_cols(16);

# The dataset: change these to scale it. The same seed always gives the same data.
CUSTOMERS = 1000
SEED = 42
AS_OF = datetime(2026, 6, 30)

## 1. Generate the dataset

Customers come from Faker, one locale per country. Orders follow `OrderPatterns`: seasonality flipped for the Southern Hemisphere, rare grills, accessories that come with a new grill, fuel matched to the grill and restocked on a cycle. Each purchase has a hidden satisfaction, and people mostly review when they are delighted or unhappy. Review text is written from a brief per review and cached by prompt.

The dataset grows in batches, as the `generate-data` command does (`uv run generate-data --help`). Asking for a `total` is idempotent, so rerunning this notebook reuses what is there. To scale up, raise `CUSTOMERS`, or move time on with `generate-data advance --to <date>`. Each batch reports what it will cost.

In [2]:
init_dataset(DEFAULT_DIR, seed=SEED, as_of=AS_OF)
writer = FoundryReviewWriter.from_env()
report = await add_customers(DEFAULT_DIR, total=CUSTOMERS, writer=writer)
await writer.aclose()
print("\n".join(report.lines() + [""] + status()))

Nothing to do: the dataset already has 1,000 customers.

Dataset at /workspaces/retail-review-knowledge-mining/data/generated: seed 42, 2023-01-01 to 2026-06-30, 1,000 customers, 2 batch(es)
     1  add-customers  2026-09-18T19:45:13  gpt-5.6-terra                    1,000 customers, 8,441 orders, 12,706 order_lines, 1,511 reviews, 1,511 review_truth
     2  fill-texts     2026-09-18T19:54:16  gpt-5.6-terra                    1,511 review_texts


Reviews are a **biased sample** of purchases: only a minority of purchases are reviewed, and the stars lean to the extremes.

In [3]:
tables = read_dataset()
purchases = tables["order_lines"].height
reviewed = tables["reviews"].height
print(f"{reviewed:,} of {purchases:,} purchases reviewed ({reviewed / purchases:.1%})")
tables["reviews"]["rating"].value_counts().sort("rating").with_columns(share=(pl.col("count") / reviewed).round(3))

1,511 of 12,706 purchases reviewed (11.9%)


rating,count,share
i8,u32,f64
1,109,0.072
2,185,0.122
3,176,0.116
4,577,0.382
5,464,0.307


## 2. Load the reviews

One row per review, joined to its text, its product and **point-in-time customer features**: tenure, lifetime revenue, orders so far and age, all computed only from what happened before the review was written. `review_key` hashes product + text, so each distinct text is classified once.

In [4]:
reviews = load_reviews()
reviews_df = reviews.collect()
print(reviews_df.shape, "-", reviews_df["review_key"].n_unique(), "distinct texts")
reviews_df.head()

(1511, 19) - 1510 distinct texts


review_id,review_key,customer_id,product_id,product_name,product_category,rating,reviewed_at,…,tenure_days,lifetime_revenue,order_count,days_since_last_order,previous_reviews,age,gender,country
str,str,str,str,str,str,i8,datetime[μs],…,f64,f64,u32,f64,u32,i32,str,str
"""R0000001010""","""c50423f1d6d2""","""C0000001""","""P014""","""SmokeRing Premium Lump Charcoal""","""consumables""",4,2023-10-30 14:48:03.730870,…,229.477118,496.8,2,5.071366,1,64,"""male""","""Costa Rica"""
"""R0000001011""","""2d73023545b7""","""C0000001""","""P015""","""WoodChips Hickory Smoking Chips""","""consumables""",2,2023-10-30 09:38:59.730870,…,229.262488,496.8,2,4.856736,0,64,"""male""","""Costa Rica"""
"""R0000001140""","""f549fa544146""","""C0000001""","""P008""","""GrillMaster Elite Tongs""","""accessories""",5,2025-10-26 21:18:26.615662,…,956.748218,1390.14,15,5.957014,2,66,"""male""","""Costa Rica"""
"""R0000001142""","""63ecd3264df7""","""C0000001""","""P016""","""FlameStarter Natural Fire Lighter""","""consumables""",5,2025-11-11 13:26:10.615662,…,972.420255,1390.14,15,21.629051,3,66,"""male""","""Costa Rica"""
"""R0000002000""","""d35dcd80053d""","""C0000002""","""P017""","""SeasonPro BBQ Rub Collection""","""consumables""",4,2023-11-28 05:21:27.375884,…,118.510231,36.2,1,15.828391,0,41,"""male""","""New Zealand"""


A customer's features move with time. Here is the customer with the most reviews: each review sees only the orders placed before it.

In [5]:
busiest = reviews_df.group_by("customer_id").len().sort("len", "customer_id", descending=[True, False])["customer_id"][0]
reviews_df.filter(pl.col("customer_id") == busiest).select(
    "reviewed_at", "product_name", "rating", pl.col("tenure_days").round(), "lifetime_revenue", "order_count", "previous_reviews"
)

reviewed_at,product_name,rating,tenure_days,lifetime_revenue,order_count,previous_reviews
datetime[μs],str,i8,f64,f64,u32,u32
2023-10-29 12:29:47.846,"""CleanBurn Pellets""",5,192.0,1343.23,5,0
2023-11-29 14:31:56.460288,"""CleanBurn Pellets""",5,223.0,1720.78,9,1
2024-02-27 14:19:13.116413,"""FlameStarter Natural Fire Lighter""",1,313.0,1887.33,11,2
2024-04-27 14:14:11.506055,"""SmokeRing Premium Lump Charcoal""",4,373.0,2091.68,14,3
2024-06-12 08:04:05.769885,"""SmokeRing Premium Lump Charcoal""",4,419.0,2176.16,15,4
2024-10-14 17:50:05.999772,"""CleanBurn Pellets""",5,543.0,2320.66,17,5
2024-11-14 11:23:33.406340,"""CleanBurn Pellets""",5,574.0,2454.83,18,6
2025-04-26 20:20:02.765150,"""CleanBurn Pellets""",5,737.0,2720.77,21,7
2026-01-22 02:10:42.680895,"""WoodChips Hickory Smoking Chips""",2,1007.0,3279.13,27,8


The **master product list**, the options Jev picks mentions from:

In [6]:
catalogue = product_catalogue()
catalogue

product_name,product_category
str,str
"""ChefsPride Stainless Steel Spatula Set""","""accessories"""
"""FlipMaster Long Handle Fork""","""accessories"""
"""GrillGuard Heat Resistant Gloves""","""accessories"""
"""GrillMaster Elite Tongs""","""accessories"""
"""HeatShield Premium Grill Cover""","""accessories"""
"""TempCheck Digital Thermometer""","""accessories"""
"""CleanBurn Pellets""","""consumables"""
"""FlameStarter Natural Fire Lighter""","""consumables"""
"""SeasonPro BBQ Rub Collection""","""consumables"""


## 3. Break reviews into sentences

In [7]:
sentences = split_sentences(reviews).collect()
print(f"{sentences.height} sentences, {sentences.select(KEY).n_unique()} distinct to classify")
sentences.select("review_id", "product_name", "sentence_index", "sentence_count", "sentence").head(12)

6781 sentences, 6779 distinct to classify


review_id,product_name,sentence_index,sentence_count,sentence
str,str,u32,u32,str
"""R0000001010""","""SmokeRing Premium Lump Charcoal""",0,3,"""Por el precio, esta bolsa de 20 lb rinde muchísimo más de lo que esperaba; los trozos vien…"
"""R0000001010""","""SmokeRing Premium Lump Charcoal""",1,3,"""La usé en dos parrilladas largas y todavía me queda suficiente para otra, así que sale mej…"
"""R0000001010""","""SmokeRing Premium Lump Charcoal""",2,3,"""Le quito una estrella porque encontré algo de polvo al fondo de la bolsa, pero la recomien…"
"""R0000001011""","""WoodChips Hickory Smoking Chips""",0,2,"""Las virutas tardaron muchísimo en empezar a humear y luego se consumieron demasiado rápido…"
"""R0000001011""","""WoodChips Hickory Smoking Chips""",1,2,"""Apenas dieron sabor a hickory a las costillas y tuve que añadir bastante más de lo esperad…"
"""R0000001140""","""GrillMaster Elite Tongs""",0,4,"""Se sienten muy sólidos y bien hechos, nada que ver con los típicos pinchos endebles que se…"
"""R0000001140""","""GrillMaster Elite Tongs""",1,4,"""El acero es grueso, el mecanismo de bloqueo funciona suave y las puntas agarran muy bien i…"
"""R0000001140""","""GrillMaster Elite Tongs""",2,4,"""Las empuñaduras no se calientan y después de varias parrilladas siguen como nuevas."""
"""R0000001140""","""GrillMaster Elite Tongs""",3,4,"""Los recomiendo totalmente por su calidad y comodidad."""


## 4. Ask Jev about every sentence

Each call sends one sentence **plus the whole review** as JSON state, and asks 26 questions at once - Jev evaluates them in parallel, so extra questions barely cost latency.

| Question | Type | Why |
|---|---|---|
| `frustration` | Score, 5 levels | How frustrated is the customer? A continuous 0-4. |
| `problem_category` | Choice | Product quality, shipping, ease of use... or `none`. |
| `mentions__<product>` × 17 | Noul each | Which catalogue products the sentence refers to. One Noul per product, because a sentence can mention several. |
| `language` | Choice | Which common language it is written in. |
| `sentiment` | Choice | positive / negative / mixed / neutral. |
| `recommendation` | Choice | Recommends / warns others off / neither. |
| `churn_risk` | Noul | Returning it, refund, switching brand, won't buy again. |
| `safety_concern` | Noul | Burns, fire, gas, unsafe food - an escalation trigger. |
| `suggestion` | Noul | An explicit product improvement idea. |
| `competitor_mention` | Noul | Mentions a product outside our range. |

The star rating and demographics are deliberately **not** sent - see the sanity check below. Answers are cached in `data/output/`, so re-running only pays for new sentences.

In [8]:
questions = build_questions(catalogue.to_dicts())
print(len(questions), "questions per sentence; question set", QUESTION_SET_VERSION)
print("frustration rubric:", *[f"  {i}: {level}" for i, level in enumerate(FRUSTRATION_LEVELS)], sep="\n")

26 questions per sentence; question set 2026-09-18.3
frustration rubric:
  0: Not frustrated: positive, neutral or purely factual.
  1: Mild: a minor gripe or slight disappointment, said calmly.
  2: Clearly frustrated: annoyed or let down by a real problem.
  3: Very frustrated: angry, feels cheated, or the problem ruined the experience.
  4: Furious: hostile or emphatic language, demands a refund, or vows never to buy again.


In [9]:
answers = await classify_sentences(sentences, catalogue, cache_path=OUTPUT / "jev_sentence_answers.parquet")

answers.select(
    pl.col("jev_model").unique().alias("model"),
    pl.col("latency_ms").mean().round().alias("mean_latency_ms"),
    pl.col("input_tokens").sum().alias("input_tokens"),
    pl.col("error").is_not_null().sum().alias("errors"),
)

332 sentence(s) to classify, 7140 cached.


Classified 332 in 19.3s; 0 failed.


model,mean_latency_ms,input_tokens,errors
str,f64,i64,u32
"""jev-1.13.0""",338.0,21280938,0


### Watch the wire

`transcript.log_to_stdout()` prints every exchange with Jev as JSON: the request body exactly as the SDK sent it (state, model alias, questions) and the response body exactly as the API returned it (the versioned model, every answer with its full probability distribution, token usage), plus the request ID and latency. The question set is printed in full once and elided after that - pass `full_questions=True` to see it every time. The API key travels in a header and is never printed.

The run above came from the cache, so this asks about two sentences afresh. To watch the whole run instead, call `transcript.log_to_stdout()` before it and delete `data/output/jev_sentence_answers.parquet`.

In [10]:
transcript.log_to_stdout()
await classify_sentences(sentences.head(2), catalogue)  # no cache_path, so these always call the API
transcript.silence()

2 sentence(s) to classify, 0 cached.


{
  "at": "2026-09-18T20:30:49.549+00:00",
  "sentence": {"review_key": "c50423f1d6d2", "sentence_index": 1},
  "request_id": "req_01a0b636f2d274e38a5f506f5e833597",
  "latency_ms": 731,
  "sent": {
    "state": {
      "task": "Classify one sentence from a customer review of a barbecue product.",
      "note": "Judge what the sentence itself says. The full review is context for resolving what 'it' or 'this' refers to.",
      "sentence": "La usé en dos parrilladas largas y todavía me queda suficiente para otra, así que sale mejor que varias marcas más caras.",
      "sentence_position": "2 of 3",
      "review": {
        "product_reviewed": "SmokeRing Premium Lump Charcoal",
        "product_category": "consumables",
        "full_text": "Por el precio, esta bolsa de 20 lb rinde muchísimo más de lo que esperaba; los trozos vienen bastante grandes y prende rápido sin dejar sabores raros en la carne. La usé en dos parrilladas largas y todavía me queda suficiente para otra, así que sale

{
  "at": "2026-09-18T20:30:49.770+00:00",
  "sentence": {"review_key": "c50423f1d6d2", "sentence_index": 0},
  "request_id": "req_01a0b636f3c77ea89c74f00f8a7ef161",
  "latency_ms": 952,
  "sent": {
    "state": {
      "task": "Classify one sentence from a customer review of a barbecue product.",
      "note": "Judge what the sentence itself says. The full review is context for resolving what 'it' or 'this' refers to.",
      "sentence": "Por el precio, esta bolsa de 20 lb rinde muchísimo más de lo que esperaba; los trozos vienen bastante grandes y prende rápido sin dejar sabores raros en la carne.",
      "sentence_position": "1 of 3",
      "review": {
        "product_reviewed": "SmokeRing Premium Lump Charcoal",
        "product_category": "consumables",
        "full_text": "Por el precio, esta bolsa de 20 lb rinde muchísimo más de lo que esperaba; los trozos vienen bastante grandes y prende rápido sin dejar sabores raros en la carne. La usé en dos parrilladas largas y todavía me

Classified 2 in 1.0s; 0 failed.


Join the answers back onto every sentence (duplicated reviews pick up the same answers) and save the flat result.

In [11]:
classified = sentences.join(answers, on=KEY, how="left")
classified.drop("answers_json").write_parquet(OUTPUT / "sentences_classified.parquet")
classified.select(
    "product_name", "sentence", pl.col("frustration").round(2), "problem_category", "sentiment", "products_mentioned", "language"
).head(15)

product_name,sentence,frustration,problem_category,sentiment,products_mentioned,language
str,str,f64,str,str,list[str],str
"""SmokeRing Premium Lump Charcoal""","""Por el precio, esta bolsa de 20 lb rinde muchísimo más de lo que esperaba; los trozos vien…",0.0,"""none""","""positive""","[""SmokeRing Premium Lump Charcoal""]","""spanish"""
"""SmokeRing Premium Lump Charcoal""","""La usé en dos parrilladas largas y todavía me queda suficiente para otra, así que sale mej…",0.0,"""none""","""positive""","[""SmokeRing Premium Lump Charcoal""]","""spanish"""
"""SmokeRing Premium Lump Charcoal""","""Le quito una estrella porque encontré algo de polvo al fondo de la bolsa, pero la recomien…",0.97,"""other""","""mixed""","[""SmokeRing Premium Lump Charcoal""]","""spanish"""
"""WoodChips Hickory Smoking Chips""","""Las virutas tardaron muchísimo en empezar a humear y luego se consumieron demasiado rápido…",1.99,"""performance""","""negative""","[""WoodChips Hickory Smoking Chips""]","""spanish"""
"""WoodChips Hickory Smoking Chips""","""Apenas dieron sabor a hickory a las costillas y tuve que añadir bastante más de lo esperad…",1.91,"""performance""","""negative""","[""WoodChips Hickory Smoking Chips""]","""spanish"""
"""GrillMaster Elite Tongs""","""Se sienten muy sólidos y bien hechos, nada que ver con los típicos pinchos endebles que se…",0.01,"""none""","""positive""","[""GrillMaster Elite Tongs""]","""spanish"""
"""GrillMaster Elite Tongs""","""El acero es grueso, el mecanismo de bloqueo funciona suave y las puntas agarran muy bien i…",0.0,"""none""","""positive""","[""GrillMaster Elite Tongs""]","""spanish"""
"""GrillMaster Elite Tongs""","""Las empuñaduras no se calientan y después de varias parrilladas siguen como nuevas.""",0.0,"""none""","""positive""","[""GrillMaster Elite Tongs""]","""spanish"""
"""GrillMaster Elite Tongs""","""Los recomiendo totalmente por su calidad y comodidad.""",0.0,"""none""","""positive""","[""GrillMaster Elite Tongs""]","""spanish"""


## 5. What the answers say

### Sanity check: does frustration track the star rating?

Jev never saw the rating, so if frustration falls as stars rise, it is reading the text rather than the number.

In [12]:
by_review = review_rollup(classified.lazy())
frustration_by_rating(by_review).collect()

rating,reviews,peak_frustration,mean_frustration
i8,u32,f64,f64
1,109,2.94,2.39
2,185,2.22,1.85
3,176,1.35,0.87
4,577,0.81,0.21
5,464,0.01,0.0


### Which problems, for which products

In [13]:
problems = problems_by_product(classified.lazy()).collect()
problems.pivot("problem_category", index="product_name", values="sentences").fill_null(0).sort("product_name")

product_name,performance,price_value,shipping_delivery,build_quality,size_capacity,ease_of_use,other,cleaning_maintenance,safety,assembly,general_dissatisfaction,customer_service
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""BBQ Baron Pellet Smoker""",2,4,0,1,9,4,0,2,0,1,0,0
"""BackYard King Compact Grill""",38,18,6,25,0,7,0,17,7,16,5,11
"""ChefsPride Stainless Steel Spatula Set""",2,4,0,15,14,27,2,1,10,0,0,0
"""CleanBurn Pellets""",39,37,13,37,2,35,5,22,0,0,0,0
"""FireMaster Pro 3000 Gas Grill""",0,0,0,0,0,0,0,0,0,3,0,0
"""FlameForge Electric Indoor Grill""",11,4,3,6,0,7,0,10,2,5,1,1
"""FlameStarter Natural Fire Lighter""",60,86,35,46,5,5,1,0,0,0,8,3
"""FlipMaster Long Handle Fork""",4,12,0,28,18,24,0,1,7,0,0,0
"""GrillGuard Heat Resistant Gloves""",16,18,0,35,9,39,0,1,21,0,0,0


In [14]:
# The most frustrating problem areas, where at least 3 reviews raise them
problems.filter(pl.col("reviews") >= 3).sort("mean_frustration", descending=True).head(10)

product_name,problem_category,sentences,reviews,mean_frustration
str,str,u32,u32,f64
"""WoodChips Hickory Smoking Chips""","""general_dissatisfaction""",14,14,3.18
"""BackYard King Compact Grill""","""price_value""",18,15,2.97
"""BackYard King Compact Grill""","""customer_service""",11,4,2.8
"""BackYard King Compact Grill""","""general_dissatisfaction""",5,5,2.69
"""WoodChips Hickory Smoking Chips""","""customer_service""",5,5,2.68
"""TurboGrill Portable Gas""","""customer_service""",9,4,2.59
"""WoodChips Hickory Smoking Chips""","""price_value""",104,66,2.48
"""TurboGrill Portable Gas""","""price_value""",5,4,2.42
"""BackYard King Compact Grill""","""assembly""",16,6,2.31


### Language

About a third of customers in non-English-speaking countries write in their own language. `text_language` is the language each review was written in; `language` is what Jev heard, per sentence. Zulu is deliberately not one of Jev's options, so South African reviews in Zulu should come back as `other`.

In [15]:
(
    classified.group_by("text_language", "language")
    .agg(sentences=pl.len(), min_confidence=pl.col("language_confidence").min())
    .sort("text_language", pl.col("sentences"), descending=[False, True])
)

text_language,language,sentences,min_confidence
str,str,u32,f64
"""chinese""","""chinese""",210,0.91
"""english""","""english""",5527,1.0
"""german""","""german""",158,1.0
"""japanese""","""japanese""",215,0.98
"""portuguese""","""portuguese""",146,0.98
"""spanish""","""spanish""",220,0.96
"""swahili""","""swahili""",136,1.0
"""zulu""","""other""",167,0.53
"""zulu""","""french""",2,1.0


### Products mentioned in reviews of *other* products

Compatibility and bundling signals - e.g. a cover reviewed alongside the grill it fits.

In [16]:
cross_mentions(classified.lazy()).collect()

product_name,also_mentions,sentence
str,str,str
"""CleanBurn Pellets""","""WoodChips Hickory Smoking Chips""","""火付きが良く、煙の出方も安定していて、スペアリブを6時間ほど低温で炊いても温度がほとんどブレませんでした。"""
"""CleanBurn Pellets""","""WoodChips Hickory Smoking Chips""","""20ポンドでこの価格ならかなりお得です。"""
"""CleanBurn Pellets""","""WoodChips Hickory Smoking Chips""","""火付きも安定していてヒッコリーの香りもしっかり出るので、リブを3回焼いても大満足でした。"""
"""CleanBurn Pellets""","""WoodChips Hickory Smoking Chips""","""ヒッコリーの香りはしっかりあるのに苦味や灰っぽさがなく、肉にきれいなスモーキーさが付きます。"""
"""CleanBurn Pellets""","""WoodChips Hickory Smoking Chips""","""ヒッコリーの香りもしっかりしていて、スペアリブを6時間スモークしたら家族に大好評でした。"""
"""CleanBurn Pellets""","""WoodChips Hickory Smoking Chips""","""ヒッコリーらしいしっかりした香りが出ますが、肉を苦くするような強さではなく、スペアリブやブリスケットにちょうどいいです。"""
"""CleanBurn Pellets""","""WoodChips Hickory Smoking Chips""","""ヒッコリーの香りも程よく、スペアリブがとてもおいしく仕上がりました。"""
"""CleanBurn Pellets""","""WoodChips Hickory Smoking Chips""","""ヒッコリーの香りはしっかり出て、豚肩ロースを6時間ほど燻したときの風味は悪くありませんでした。"""
"""SmokeRing Premium Lump Charcoal""","""FlameStarter Natural Fire Lighter""","""Os pedaços vieram em bom tamanho, com pouca poeira no fundo do saco, e acendem rápido usan…"


### Escalations: safety concerns and churn risk

In [17]:
flagged(classified.lazy(), "safety_concern").collect()

product_name,sentence,safety_concern,occurrences,frustration
str,str,f64,u32,f64
"""BackYard King Compact Grill""","""Grease dripped past the tray and collected underneath the firebox, which is exactly the ki…",0.98,1,2.14
"""GrillGuard Heat Resistant Gloves""","""I grabbed a grate for maybe two seconds and felt a sharp burn across my palm even though t…",0.98,1,2.06
"""BackYard King Compact Grill""","""The electronic ignition kept clicking after the burner lit, and once I smelled gas around …",0.98,1,2.02
"""FlipMaster Long Handle Fork""","""One prong slipped out of the meat and sent hot grease splattering onto my wrist.""",0.98,1,2.06
"""GrillGuard Heat Resistant Gloves""","""I bought these specifically to keep my hands safe around my smoker, and instead I ended up…",0.97,1,2.41
"""GrillGuard Heat Resistant Gloves""","""These gloves are advertised for handling hot grates, but they let heat through far too qui…",0.96,1,2.12
"""FlipMaster Long Handle Fork""","""This fork feels downright unsafe to use around a hot grill.""",0.96,1,2.23
"""BackYard King Compact Grill""","""This grill felt unsafe from the first weekend.""",0.96,1,2.31
"""GrillGuard Heat Resistant Gloves""","""The silicone grip became slippery once it had a little grease on it, and I nearly dropped …",0.95,1,2.26


In [18]:
flagged(classified.lazy(), "churn_risk").collect()

product_name,sentence,churn_risk,occurrences,frustration
str,str,f64,u32,f64
"""WoodChips Hickory Smoking Chips""","""I feel completely ripped off and will not be buying these again.""",0.96,2,3.7
"""WoodChips Hickory Smoking Chips""","""I will not be buying this brand again.""",0.96,1,3.74
"""CleanBurn Pellets""","""My smoker kept struggling to maintain temperature because the fines were clogging the hopp…",0.96,1,3.04
"""WoodChips Hickory Smoking Chips""","""Won’t be buying this brand again.""",0.96,1,3.33
"""SmokeRing Premium Lump Charcoal""","""I feel completely cheated and will not be buying this again.""",0.96,1,3.53
"""FlameStarter Natural Fire Lighter""","""I would not buy these again.""",0.96,1,2.7
"""FlameForge Electric Indoor Grill""","""I feel completely cheated for the price and returned it after one miserable weekend.""",0.96,1,3.01
"""WoodChips Hickory Smoking Chips""","""Absolutely disappointed and I will not be buying this brand again.""",0.96,1,3.92
"""WoodChips Hickory Smoking Chips""","""For the price, I feel completely ripped off and went back to my usual brand.""",0.96,1,3.01


### Free product ideas: explicit suggestions

In [19]:
flagged(classified.lazy(), "suggestion").collect()

product_name,sentence,suggestion,occurrences,frustration
str,str,f64,u32,f64
"""GrillGuard Heat Resistant Gloves""","""I just wish the cuffs were a little longer for reaching deeper into the grill.""",0.98,1,1.0
"""SeasonPro BBQ Rub Collection""","""I only wish the Carolina tang rub came in a slightly larger tin, since we went through tha…",0.98,1,0.98
"""ChefsPride Stainless Steel Spatula Set""","""I’d definitely recommend the set; I only wish the handles had hanging loops for easier sto…",0.97,1,0.97
"""FlipMaster Long Handle Fork""","""Die Zinken könnten etwas schärfer und minimal weiter auseinander stehen, damit sie besser …",0.97,1,0.99
"""ChefsPride Stainless Steel Spatula Set""","""I took off one star because I wish the handles were a little longer for my larger barbecue…",0.97,1,1.0
"""HeatShield Premium Grill Cover""","""唯一希望是提手位置再明显一点，晚上收盖时会更好找。""",0.97,1,0.99
"""FlameStarter Natural Fire Lighter""","""Better packaging and firmer cubes would make these much more reliable.""",0.97,1,1.14
"""ChefsPride Stainless Steel Spatula Set""","""The thin flipper is great for pancakes and eggs, though I wish the handle on that one were…",0.97,1,0.99
"""FlameStarter Natural Fire Lighter""","""Lost one star only because I would prefer slightly larger cubes for bigger charcoal loads.""",0.97,1,0.99


### Competitors and previous products

In [20]:
flagged(classified.lazy(), "competitor_mention").collect()

product_name,sentence,competitor_mention,occurrences,frustration
str,str,f64,u32,f64
"""FlameStarter Natural Fire Lighter""","""I’ll go back to the cheaper firelighters from the supermarket.""",0.96,1,1.67
"""GrillGuard Heat Resistant Gloves""","""I’ve bought cheaper hardware-store gloves that performed better and lasted longer.""",0.95,1,1.95
"""FlameStarter Natural Fire Lighter""","""Cheaper supermarket fire lighters have lasted me longer and worked almost as well.""",0.95,1,1.3
"""BBQ Baron Pellet Smoker""","""The meat probe reads a few degrees off compared with my ThermoPro.""",0.95,1,1.0
"""FlameStarter Natural Fire Lighter""","""Cheaper supermarket fire lighters have lasted longer for me and worked just as reliably.""",0.95,1,1.47
"""FlameStarter Natural Fire Lighter""","""I usually needed two or three lighters where one firelighter from my previous brand did th…",0.94,1,1.64
"""FlameStarter Natural Fire Lighter""","""Great value for money compared with the supermarket firelighters I was buying before.""",0.93,1,0.0
"""FlipMaster Long Handle Fork""","""It feels sturdier than the cheap fork I picked up at the grocery store last summer.""",0.93,1,0.0
"""TempCheck Digital Thermometer""","""Both probes have given me accurate readings against my old instant-read thermometer.""",0.93,1,0.0


### Low confidence: route to a person

Jev returns a confidence with every Choice. Where it could not separate the problem categories, that is a signal to send the sentence for human review rather than trust the label.

In [21]:
needs_review(classified.lazy(), min_confidence=0.6).collect()

product_name,sentence,problem_category,problem_category_confidence
str,str,str,f64
"""SmokeRing Premium Lump Charcoal""","""Once I opened it, I found way too many tiny pieces and crumbs for what is supposed to be p…","""build_quality""",0.2
"""WoodChips Hickory Smoking Chips""","""The bag also had far more dust and tiny fragments than usable chips.""","""performance""",0.2
"""WoodChips Hickory Smoking Chips""","""For the price they work well enough, but I would prefer a more consistent chip size.""","""performance""",0.2
"""HeatShield Premium Grill Cover""","""Die Passform ist für einen Full-Size-Grill in Ordnung, aber bei Wind würde ich mir zusätzl…","""size_capacity""",0.21
"""SmokeRing Premium Lump Charcoal""","""There was a lot more small pieces and dust at the bottom than I expected from a premium lu…","""build_quality""",0.21
"""SmokeRing Premium Lump Charcoal""","""There was also more small rubble at the bottom of the bag than I wanted for premium lump.""","""price_value""",0.22
"""SmokeRing Premium Lump Charcoal""","""Took off one star because there were a few small bits near the bottom, but overall it does…","""other""",0.23
"""SmokeRing Premium Lump Charcoal""","""I knocked off one star because I found a couple of small pieces near the bottom, but I’d d…","""size_capacity""",0.23
"""SmokeRing Premium Lump Charcoal""","""What was left inside was mostly undersized pieces and crumbled junk, not premium lump char…","""performance""",0.24


### Most frustrated reviews

In [22]:
by_review.sort("peak_frustration", descending=True).select(
    "rating", "product_name", pl.col("peak_frustration").round(2), "problems", "churn_risk", "safety_concern", "review_text"
).unique("review_text", maintain_order=True).head(10).collect()

rating,product_name,peak_frustration,problems,churn_risk,safety_concern,review_text
i8,str,f64,list[str],bool,bool,str
1,"""WoodChips Hickory Smoking Chips""",3.97,"[""build_quality"", ""general_dissatisfaction"", … ""shipping_delivery""]",true,false,"""这袋所谓山核桃木屑的品质太差了，打开后里面一半都是粉末和碎渣，根本不像正常的木片。大块木屑又潮又软，有几块甚至像是发霉变黑了。放进烟盒没多久就闷灭，只冒出一股刺鼻的焦糊味，完全没有…"
1,"""WoodChips Hickory Smoking Chips""",3.92,"[""general_dissatisfaction"", ""performance"", … ""shipping_delivery""]",true,false,"""The bag was full of tiny crumbs and dust instead of proper smoking chips. Nearly half of i…"
2,"""WoodChips Hickory Smoking Chips""",3.91,"[""ease_of_use"", ""performance"", ""price_value""]",true,false,"""ヒッコリーの香りを期待して買ったのに、ほとんど煙が出ず風味も全然付きませんでした。説明どおりに水に浸してから炭火グリルに入れましたが、すぐに黒く焦げて終わりです。少し煙が出たと思っ…"
1,"""TurboGrill Portable Gas""",3.9,"[""customer_service"", ""general_dissatisfaction"", ""shipping_delivery""]",true,false,"""Paid extra for expedited shipping so I’d have this grill for a tailgate, and it showed up …"
2,"""WoodChips Hickory Smoking Chips""",3.81,"[""performance"", ""price_value""]",true,false,"""£9.99 for what turned out to be a tiny bag of mostly dust and splinters is ridiculous. I g…"
1,"""WoodChips Hickory Smoking Chips""",3.75,"[""customer_service"", ""general_dissatisfaction"", … ""shipping_delivery""]",true,false,"""Ngilinde lezi zinkuni ze-hickory amasonto amabili, nakuba ekhasini bekuthi ukulethwa kuzot…"
2,"""WoodChips Hickory Smoking Chips""",3.74,"[""build_quality"", ""general_dissatisfaction"", … ""shipping_delivery""]",true,false,"""Opened the bag and found more dust and tiny splinters than usable smoking chips. The piece…"
1,"""WoodChips Hickory Smoking Chips""",3.73,"[""general_dissatisfaction"", ""performance"", … ""size_capacity""]",true,false,"""Ngikhokhe imali eningi kakhulu ngalesi sikhwama sama-hickory chips ngilindele ukuthi sizoh…"
1,"""WoodChips Hickory Smoking Chips""",3.73,"[""build_quality"", ""customer_service"", … ""shipping_delivery""]",true,false,"""Ordered these hickory chips for a cookout and paid extra for expedited shipping, but the b…"
